# Notebook 18 visual report
Purpose: portable diagnostic,training-work and paired-quality figures. Inputs: committed18receipts only; missing values remain missing. Outputs: report.html,PNGfigures,result.json. Runtime: seconds. Checkpoints: no model work; source receipts remain authoritative. Resume: rerender unchanged receipts; no predictive data reads or training.

In [ ]:
from pathlib import Path
import base64,html,json,os,time

def render(root,artifact,result,diagnostic=None,development=None,training=None):
    """Portable measured-evidence visuals; never replaces an absent result with zero."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True);figures=[]
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    colors={'vit':'#315f8c','candidate':'#c27228','raw':'#687881','centered':'#315f8c'}
    prefix='Software fixture — 'if result.get('fixture_source')else''
    def save(fig,name):
        fig.savefig(artifact/name,dpi=145,bbox_inches='tight');plt.close(fig);figures.append(name)
    if diagnostic and diagnostic.get('rows'):
        rows=[r for r in diagnostic['rows']if r.get('status')in('COMPLETE','MEASURED','PASS')and r.get('measurements')]
        if rows:
            ranks=sorted({int(r['rank'])for r in rows});layers=sorted({int(r['layer'])for r in rows})
            fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
            for layer in layers:
                for name in ('raw','centered'):
                    for ax,key in zip(axes,('relative_projected_output_l2','max_relative_shared_parameter_gradient_l2')):
                        med=[];lo=[];hi=[]
                        for rank in ranks:
                            values=[r['measurements'][name].get(key,np.nan)for r in rows if r['rank']==rank and r['layer']==layer]
                            values=[v for v in values if v is not None and np.isfinite(v)]
                            med.append(float(np.median(values))if values else np.nan);lo.append(min(values)if values else np.nan);hi.append(max(values)if values else np.nan)
                        ax.plot(ranks,med,'o-'if layer==0 else's--',color=colors[name],label=name+', layer'+str(layer))
                        ax.fill_between(ranks,lo,hi,color=colors[name],alpha=.08)
            for ax,title in zip(axes,('Projected attention output deviation','Worst shared-parameter gradient deviation')):
                ax.set_title(title);ax.set_xlabel('Random features');ax.set_ylabel('Relative L2 error');ax.set_xticks(ranks);ax.set_ylim(bottom=0);ax.grid(alpha=.2);ax.legend(fontsize=8)
            fig.suptitle(prefix+'Frozen TRAIN-only ViT diagnostic; bands show observed basis-seed range')
            save(fig,'diagnostic_fidelity.png')
            unique={r['layer']:r.get('token_statistics',{})for r in rows}
            if all('pair_sum_squared_over_tau'in s for s in unique.values()):
                fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
                for layer,s in unique.items():
                    for ax,key in zip(axes,('pair_sum_squared_over_tau','centered_pair_sum_squared_over_tau')):
                        vals=s[key];pairs=[('p'+str(int(100*level)),value)for level,value in zip(vals['quantile_levels'],vals['values'])]
                        ax.plot(range(len(pairs)),[v for k,v in pairs],'o-',label='layer'+str(layer));ax.set_xticks(range(len(pairs)),[k for k,v in pairs],rotation=45)
                for ax,title,ylabel in zip(axes,('Raw Gaussian-feature variance indicator','Centered variance indicator'),('||q+k||² / τ','||q+k−2c||² / τ')):
                    ax.set_title(title);ax.set_ylabel(ylabel);ax.set_ylim(bottom=0);ax.legend();ax.grid(alpha=.2)
                fig.suptitle(prefix+'Measured TRAIN-token geometry; this indicator alone is not a causal explanation')
                save(fig,'diagnostic_geometry.png')
    def quality_plot(quality,name,title,initial=False):
        pairs=quality.get('paired_results',[])
        if not pairs:return
        fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
        for index,key,color,label in [(0,'vit_percent',colors['vit'],'ViT'),(1,'candidate_percent',colors['candidate'],'Learned kernel')]:
            values=[p[key]for p in pairs];axes[0].scatter([index]*len(values),values,facecolors='none',edgecolors=color);axes[0].scatter([index],[np.mean(values)],marker='D',s=65,color=color)
            if initial:
                ikey='vit_initial_percent'if index==0 else'candidate_initial_percent'
                for p in pairs:axes[0].plot([index-.12,index],[p[ikey],p[key]],color=color,alpha=.35)
                axes[0].scatter([index-.12]*len(values),[p[ikey]for p in pairs],marker='x',color=color,label=label+' initial')
        displayed=[float(p[key])for p in pairs for key in ('vit_percent','candidate_percent')]
        if initial:displayed.extend(float(p[key])for p in pairs for key in ('vit_initial_percent','candidate_initial_percent'))
        axes[0].set_xticks([0,1],['ViT','Learned kernel']);axes[0].set_ylim(0,max(1.,1.08*max(displayed)));axes[0].set_ylabel('Balanced top-1 accuracy (%)')
        if initial:axes[0].legend(fontsize=8)
        deltas=[p['delta_pp']for p in pairs];axes[1].scatter(range(1,len(pairs)+1),deltas,color=colors['candidate']);axes[1].axhline(0,color='#555');axes[1].axhline(quality['mean_delta_pp'],color=colors['candidate'],linestyle=':',label='Mean paired effect')
        ci=quality.get('raw_ci95_pp')
        if ci is not None:axes[1].axhspan(ci[0],ci[1],color=colors['candidate'],alpha=.15,label='95%paired-t mean interval')
        else:axes[1].text(.02,.98,'Paired-t interval undefined',transform=axes[1].transAxes,va='top')
        axes[1].set_xticks(range(1,len(pairs)+1));axes[1].set_xlabel('Independent paired training seed');axes[1].set_ylabel('Learned − ViT (percentage points)');axes[1].legend(fontsize=8)
        fig.suptitle(prefix+title);save(fig,name)
    if development:quality_plot(development,'development_quality.png','Secondary development results; initial-to-final lines; no model selection',True)
    if result.get('paired_results'):quality_plot(result,'official_quality.png','Primary predeclared 5,050-image official subset; eight paired training seeds')
    if training and training.get('arms'):
        arms=list(training['arms'].values());groups={name:[r for r in arms if ('vit'if r.get('variant')=='vit'or r.get('arm_id','').endswith('-vit')else'candidate')==name]for name in('vit','candidate')}
        fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
        for i,name in enumerate(('vit','candidate')):
            rows=groups[name]
            for ax,key,label in zip(axes,('actual_updates','active_wall_seconds'),('Successful optimizer updates','Active training wall seconds')):
                values=[r[key]for r in rows if r.get(key)is not None]
                if values:ax.scatter([i]*len(values),values,facecolors='none',edgecolors=colors[name]);ax.scatter([i],[np.mean(values)],marker='D',s=65,color=colors[name])
                ax.set_ylabel(label);ax.set_xticks([0,1],['ViT','Learned kernel']);ax.set_ylim(bottom=0);ax.grid(axis='y',alpha=.2)
        fig.suptitle(prefix+'Actual complete-training work under equal 1,800-second active budgets')
        save(fig,'training_work.png')
    body=['<!doctype html><html><head><meta charset="utf-8"><title>Learned kernel JEPA study</title><style>body{font:16px system-ui;max-width:1180px;margin:32px auto;padding:0 24px;color:#172330}img{width:100%;height:auto}pre{white-space:pre-wrap;word-break:break-word;font:12px monospace}p{line-height:1.55}code{background:#f1f3f5;padding:2px 5px}</style></head><body>',
        '<h1>Learned-kernel JEPA: diagnostic and paired learning</h1>',
        '<p><strong>Status: '+html.escape(str(result.get('status','UNKNOWN')))+'.</strong> '+html.escape(str(result.get('reason',result.get('phase',result.get('decision','')))))+'</p>',
        '<p>One predeclared candidate; eight fresh paired training seeds; equal 1,800 active-wall seconds per arm; fixed 12-hour maximum. Candidate adds 98,328 online parameters. Shared original weights, masking, predictor and objective remain paired. Teacher fidelity and development accuracy do not select which final models are scored.</p>',
        '<p>Official data stays closed until all registered weights and certified training-only probes are sealed. The 5,050-image subset is one fixed population; training seeds are the inferential units. A positive claim requires mean gain ≥0.5 percentage points and the paired 95% interval above zero. UNKNOWN stays missing. This is an FMM-inspired learned summary operator, not an exact ViT or classical multipole claim.</p>']
    if result.get('fixture_source'):body.append('<p><strong>SOFTWARE VALIDATION ONLY. Synthetic fixtures are not research results.</strong></p>')
    for name in figures:
        body.append('<figure><img alt="'+html.escape(name)+'"src="data:image/png;base64,'+base64.b64encode((artifact/name).read_bytes()).decode()+'"></figure>')
    audit={'result':result,'diagnostic':diagnostic,'development':development,'training':training}
    body.append('<details><summary>Inspect source-backed records and limitations</summary><pre>'+html.escape(json.dumps(audit,indent=2,allow_nan=False,default=str))+'</pre></details></body></html>')
    for filename,text in [('report.html','\n'.join(body)),('result.json',json.dumps(result,indent=2,allow_nan=False,default=str))]:
        path=artifact/filename;tmp=artifact/(filename+'.tmp.'+str(os.getpid()))
        with tmp.open('w')as handle:handle.write(text);handle.flush();os.fsync(handle.fileno())
        os.replace(tmp,path)
    return {'report':str(artifact/'report.html'),'figures':figures,'status':result.get('status','UNKNOWN')}
